In [1]:
from explore_import import  *
import ionbot_preprocess as io
import data_preprocess as dt
import hpp_checker as hpp
from Download_UnimodDB import *

from pyteomics import mass
import itertools 
import time
import re
import ast
from scipy.spatial import distance
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
with open('combined_first_datasets.pickle', 'rb') as handle:
    combined_first_datasets=pickle.load(handle)

In [14]:
insituPEP=pd.read_csv("./../insitu_peptides.csv")

In [3]:
def Download_Unimod_Dict_names():
    unimod = Download_UnimodDB()
    condensed_unimod = {}
    for uni_id,df in unimod.groupby("unimod_id").__iter__():
        condensed_unimod[uni_id] = {}
        condensed_unimod[uni_id]['residues'] = "".join([_ for _ in df.residue if '-' not in _])
        condensed_unimod[uni_id]['mono_mass'] = [_ for _ in set(df.mono_mass)][0]
        condensed_unimod[uni_id]['code_name'] = [_ for _ in set(df.code_name)][0]
        condensed_unimod[uni_id]['full_name'] = [_ for _ in set(df.full_name)][0]
        
    return condensed_unimod

In [4]:
unimod = Download_Unimod_Dict_names()
unimod_df=pd.DataFrame.from_dict(unimod, orient='index')
#mark substitutions
unimod_df["isSubstitution"]=unimod_df.full_name.apply(lambda x: "substitution" in x)
unimod_df_subs=unimod_df[unimod_df["isSubstitution"]]

In [5]:
#get the data
comb_datasets=pd.DataFrame()
for dataset_name,subdict in combined_first_datasets.items():
    df=subdict["openprot"]
    df["dataset"]=dataset_name
    comb_datasets=pd.concat([comb_datasets,df])
comb_datasets.reset_index(drop=True,inplace=True)
comb_datasets["peptide_length"]=comb_datasets.database_peptide.apply(lambda x: len(x))

In [8]:
#how many isoSAAV at all by class
(comb_datasets[comb_datasets.isSubstitution]["peptide_class"].value_counts()/len(comb_datasets))*100

peptide_class
shared_btw_can_noncan    15.673863
shared_in_Canon           9.617715
unique_to_Canon           1.842626
unique_to_Noncanon        0.866429
shared_in_Noncanon        0.106920
Name: count, dtype: float64

In [9]:
#how many isoSAAV inside each class
(comb_datasets[comb_datasets.isSubstitution]["peptide_class"].value_counts()/comb_datasets["peptide_class"].value_counts())*100

peptide_class
shared_btw_can_noncan    27.003063
shared_in_Canon          26.793803
unique_to_Canon          40.108793
unique_to_Noncanon       66.287512
shared_in_Noncanon       67.330677
Name: count, dtype: float64

In [10]:
#check that substitution location is on expected residue

comb_datasets["isSubAA"]=np.nan
comb_datasets["isSubMass"]=np.nan
for i, row in comb_datasets[comb_datasets.isSubstitution].iterrows():
    modifications_masses=row.modifications_masses
    for pos, aa, mass in modifications_masses:
        info=unimod_df_subs[unimod_df_subs.mono_mass==mass]
        if len(info[info.residues.str.contains(aa)]):
            comb_datasets.loc[i,["isSubAA","isSubMass"]]=[aa,mass]

In [11]:
# X where PTM aa corresponds to expected for isoSAAV
(comb_datasets[comb_datasets.isSubstitution].isSubAA.isna().value_counts()/len(comb_datasets[comb_datasets.isSubstitution]))*100

isSubAA
True     80.456924
False    19.543076
Name: count, dtype: float64

In [ ]:
#however, we are not sure if location is correct, so get mass and compare it to canonical peptides of same mass with 1-aa distance
# mass is already available in column peptide_mass (includes mass shift)

mass_error=0.000100
for i, row in comb_datasets[comb_datasets.isSubstitution].iterrows():
    peptide_length=len(row.database_peptide)
    peptide_mass=row.peptide_mass
    candidates=insituPEP.loc[((insituPEP.protein_class=="Canon")&(insituPEP.peptide_length==peptide_length)&(abs(insituPEP.peptide_mass-peptide_mass)<=mass_error))]
    candidates.drop_duplicates("peptide",inplace=True)
    #check for 1 aa distance
    ham=1/peptide_length
    candidates["1_aa_dist"]=candidates.peptide.apply(lambda x: distance.hamming(list(x),list(row.database_peptide))<=ham)
    comb_datasets.loc[i,["1_aa_dist"]]="|".join(candidates[candidates["1_aa_dist"]].peptide.tolist()) if len(candidates[candidates["1_aa_dist"]])>0 else False

/tmp/ipykernel_3013492/3437720943.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates.drop_duplicates("peptide",inplace=True)
/tmp/ipykernel_3013492/3437720943.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates["1_aa_dist"]=candidates.peptide.apply(lambda x: distance.hamming(list(x),list(row.database_peptide))<=ham)
/tmp/ipykernel_3013492/3437720943.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/i